# Decomposition and Stationarity

This notebook evaluates trend, seasonality, integration, and structural break behavior in total international arrivals.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, TARGET_COLUMNS, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style

set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## Seasonal Decomposition

Observation: the log series contains a persistent trend and stable monthly pattern before the pandemic. Statistical implication: additive decomposition on logs is appropriate for separating proportional seasonal effects. Tourism implication: predictable seasonal pressure should be planned separately from crisis-driven demand changes.

In [ ]:
series = np.log(df["international_arrivals"])
decomp = seasonal_decompose(series, model="additive", period=12, extrapolate_trend="freq")
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
for ax, (name, values) in zip(axes, [("Observed", series), ("Trend", decomp.trend), ("Seasonal", decomp.seasonal), ("Residual", decomp.resid)]):
    ax.plot(values.index, values.values, color=SERIES_COLORS["international_arrivals"])
    ax.set_ylabel(name)
axes[0].set_title("Additive Decomposition of Log International Arrivals")
save_figure(fig, FIGURES / "08_decomposition.png")
plt.show()

## Unit Root Tests

Observation: stationarity tests distinguish log levels from log differences. Statistical implication: differencing is needed for stable mean dynamics. Tourism implication: forecasts should focus on growth dynamics rather than assuming fixed levels.

In [ ]:
rows = []
for name, values in [("log level", series.dropna()), ("first log difference", series.diff().dropna())]:
    adf = adfuller(values, autolag="AIC")
    try:
        kpss_result = kpss(values, regression="c", nlags="auto")
        kpss_p = kpss_result[1]
    except Exception:
        kpss_p = np.nan
    rows.append({"series": name, "ADF p-value": adf[1], "KPSS p-value": kpss_p})
stationarity = pd.DataFrame(rows)
stationarity.to_csv(TABLES / "stationarity_tests.csv", index=False)
stationarity

## ACF and PACF

Observation: autocorrelation remains visible after first differencing, especially at seasonal lags. Statistical implication: SARIMA terms are justified. Tourism implication: recurring calendar effects improve forecast discipline for operational planning.

In [ ]:
diff = series.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
plot_acf(diff, lags=36, ax=axes[0], color=SERIES_COLORS["international_arrivals"], zero=False)
plot_pacf(diff, lags=36, ax=axes[1], color=SERIES_COLORS["international_arrivals"], zero=False, method="ywm")
axes[0].set_title("ACF of Log-Differenced Arrivals")
axes[1].set_title("PACF of Log-Differenced Arrivals")
for ax in axes:
    ax.grid(False)
save_figure(fig, FIGURES / "09_acf_pacf.png")
plt.show()

## Log Transformation and Structural Break

Observation: the log transformation stabilizes proportional variation but cannot remove the COVID break. Statistical implication: transformation handles scale, while model evaluation must still account for regime change. Tourism implication: post-pandemic uncertainty should remain central in forecast interpretation.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(df.index, df["international_arrivals"], color=SERIES_COLORS["international_arrivals"])
axes[0].set_title("Level Series")
axes[1].plot(series.index, series, color=SERIES_COLORS["international_arrivals"])
axes[1].set_title("Log Series")
for ax in axes:
    annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening"])
save_figure(fig, FIGURES / "11_log_transformation_comparison.png")
plt.show()